# $\zeta(3)$: historical constructions and new result

The CMF examples sample every 10 depths through depth 3000; the Dvornicich–Viola
staircase samples $m=10,20,\ldots,300$, plotted with $N=m$; Giraudin's direction is
computed at depth 1 only.
Each example prints its final $\delta_N$; the figure at the end plots
$\delta_N$ against $N$, where
$$D_N=-\log_{10}\left|\zeta(3)-\frac{p_N}{q_N}\right|,
\qquad \delta_N=\frac{D_N}{\log_{10}q_N}-1.$$
Here $p_N/q_N$ is reduced. Accuracy digits measure absolute approximation error.

Run the cells in order (Python 3.11+). The CMF encodings come from the supplied
exploratory notebooks and local supplementary material. Published bounds and the
finite-depth $\delta_N$ values are different quantities; the computations below do
not by themselves prove a published irrationality-measure bound.

The explicit Apéry and Zudilin (2002) recoveries below sample source indices
10,20,...,1000. Their CMF walk length is N-2; earlier sections retain their
stated sampling conventions. Only the changed recovery cells were rerun.

Dauguet–Zudilin (2014) also uses source indices through 1000, equal to its walk depths.


In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

Each example constructs a CMF, walks a trajectory with ramanujantools, reads
$p_N/q_N$ off the first column $(s_0,s_1,\ldots)$ of the walk, and computes
$\delta_N$. The walks and the approximants are exact.

In [2]:
from cmfs import apery_seeds, zudilin_zeta23_seeds, zudilin_zeta23_readout
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx, fmpq, fmpq_mat
from ramanujantools import Position
from ramanujantools.cmf.meijer_g import MeijerG
from ramanujantools.cmf.pfq import pFq
from ramanujantools.flint_core.numeric_matrix import NumericMatrix

from cmfs import axis_steps

sys.set_int_max_str_digits(0)
with ctx.workdps(301_000):  # q_N reaches about 130,000 digits at depth 3000
    ZETA3 = arb(3).zeta()
    ZETA4 = arb(4).zeta()
    JOINT = 15 * ZETA3 + ZETA4
DEPTHS = list(range(10, 3001, 10))
n = sp.Symbol("n")
x0, x1, x2, x3, y0, y1, y2 = sp.symbols("x0 x1 x2 x3 y0 y1 y2")
a0, a1, a2, a3, b0, b1, b2, b3 = sp.symbols("a0:4 b0:4")
results = {}


def delta(approximant, value=ZETA3):
    """delta with |value - p/q| = q^-(1 + delta), for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):  # enough bits to resolve |value - p/q|
        error = abs(value - arb(p) / q)
        assert error > 0
        return float((-1 - error.log() / arb(q).log()).mid())

## 1. Apéry (1978 announcement; 1979 publication): exact source sequences

[Original paper, pp. 12–13](https://www.numdam.org/article/AST_1979__61__11_0.pdf).
Both sequences satisfy
$$(N+1)^3u_{N+1}-(34N^3+51N^2+27N+5)u_N+N^3u_{N-1}=0,$$
with $(a_0,a_1)=(1,5)$ and $(b_0,b_1)=(0,6)$;
the approximants are $b_N/a_N$. The recurrence normalization is also recovered at
$\alpha=0$ in [Zudilin (2003)](https://arxiv.org/abs/math/0307058).
Apéry's asymptotic measure is
$\mu_0=2\log\lambda/(\log\lambda-3)\approx 13.41782$, $\lambda=17+12\sqrt2$.

**Full CMF:** ${}_4F_3(-N,-N,N+1,N+1;1,1,1;1)$, rank 3, 7 independent parameter axes.
**Start:** $(-2,-2,3,3;1,1,1)$ (source index 2); **direction:** $(-1,-1,1,1;0,0,0)$.
With $W$ the walk of length $N-2$, the exact readout is $(PW)_0/(QW)_0$, where
$$P=(351/4,2683/24,0),\quad Q=(73,93,0).$$
The denominator is the terminating binomial sum
$a_N=\sum_{k=0}^N\binom Nk^2\binom{N+k}k^{2}$.
The full-field trajectory satisfies the displayed recurrence as a symbolic identity;
its first two source values fix the numerator and denominator rows.

`local_checks/verify_apery_zudilin_zeta23.py` checks that identity, the binomial sums at small
indices, exact source fractions at every sampled index through 1000, and all axis
pairs at two off-trajectory rational points. Two Arb precisions give
$\delta_{1000}=0.085098239651$. This measured exponent is distinct from the published bound.


In [ ]:
cmf = pFq(4, 3, 1)
axes = sp.symbols("x0:4") + sp.symbols("y0:3")
start = dict(zip(axes, [-2, -2] + [3]*2 + [1]*3))
trajectory = dict(zip(axes, [-1, -1] + [1]*2 + [0]*3))
depths = list(range(10, 1001, 10))  # source indices
P, Q = apery_seeds(3)
walks = cmf.walk(trajectory, [N - 2 for N in depths], start)
approximants = [(P*W)[0]/(Q*W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Apéry (1979), exact source index"] = (depths, deltas)
print(f"delta at source index 1000 (998 steps): {deltas[-1]:.12f}")


delta at source index 1000 (998 steps): 0.085098239651


## 2. Zudilin (2002): simultaneous zeta(2), zeta(3) approximations

[Original paper, Section 3, equation (16) and Theorem 4](https://arxiv.org/abs/math/0206178).
This is a distinct third-order construction. Its proven convergence rate is
$$\lim_{N\to\infty}\frac1N\log|\zeta(s)-p_N^{(s)}/q_N|
=\log|\mu_2|-\log|\mu_3|,$$
where $\mu_{1,2}=0.07260980\ldots\pm0.25981363\ldots i$,
$\mu_3=219.85478039\ldots$ solve $\mu^3-220\mu^2+32\mu-16=0$.
No irrationality-measure bound is claimed for this recovery.

**Full CMF:** rank-3 ${}_4F_3(1)$ with all seven independent axes. Define
$$F_N(z)={}_4F_3(N+1,N+1,N+1,N+1;2N+2,2N+2,2N+2;z),
\quad C_N=\frac{N!^6}{(2N+1)!^3},\quad\theta=z\frac{d}{dz}.$$
The paper's auxiliary series are exactly $-C_NF_N(1)$ and
$C_N[(\theta+N+1)F_N]_{z=1}$ after the shift $k=m+N+1$.
The state is $(F,\theta F,\theta^2F)$.
**Start:** $(3,3,3,3;6,6,6)$, source index 2;
**direction:** $(1,1,1,1;2,2,2)$; **walk length:** $N-2$.

Exact seed rows in `cmfs.py` come from partial fractions of
$$\frac{(5!)^3}{(2!)^4}\frac{(m+1)(m+2)}{(m+3)^3(m+4)^3(m+5)^3}.$$
Write $F_N(1)=A_0+A_2\zeta(2)+A_3\zeta(3)$ and
$[(\theta+N+1)F_N]_{z=1}=B_0+B_2\zeta(2)+B_3\zeta(3)$.
With $D=A_3B_2-B_3A_2$, the two exact readouts are
$$\frac{p_N^{(2)}}{q_N}=\frac{B_3A_0-A_3B_0}{D},\qquad
\frac{p_N^{(3)}}{q_N}=\frac{A_2B_0-B_2A_0}{D}.$$
Multiplication by $C_N^2$ reproduces the source normalization:
$q=(1,14,978,\ldots)$, $p^{(2)}=(0,23,6435/4,\ldots)$,
$p^{(3)}=(0,17,9405/8,\ldots)$.

The verification script proves recurrence (16) symbolically for the normalized
two-by-two minors of the full-field walk, and checks the normalized forms at source indices 2–9,
the recurrence (16) at every sampled index through 1000, and both precision runs.
The $\zeta(3)$ readout gives $\delta_{1000}=-0.038940836181$; this is a negative finite-depth approximation exponent.


In [ ]:
cmf = pFq(4, 3, 1)
axes = sp.symbols("x0:4") + sp.symbols("y0:3")
start = dict(zip(axes, [3]*4 + [6]*3))
trajectory = dict(zip(axes, [1]*4 + [2]*3))
depths = list(range(10, 1001, 10))  # source indices
seeds = zudilin_zeta23_seeds()
walks = cmf.walk(trajectory, [N - 2 for N in depths], start)
approximants = [zudilin_zeta23_readout(W, N, seeds, 3) for N, W in zip(depths, walks)]
deltas = [delta(r) for r in approximants]
results["Zudilin (2002), simultaneous"] = (depths, deltas)
print(f"delta at source index 1000 (998 steps): {deltas[-1]:.12f}")


delta at source index 1000 (998 steps): -0.038940836181


## 3. Dauguet–Zudilin (2014): simultaneous construction

S. Dauguet and W. Zudilin, [*On simultaneous diophantine approximations to
$\zeta(2)$ and $\zeta(3)$*, Section 3, equations (25)–(30)](https://arxiv.org/abs/1401.5322).
Their Theorem 1 gives the specialized simultaneous bound
$s_{\tau_0}(\zeta(2),\zeta(3))\le s_0$, with
$\tau_0=0.899668635\ldots$ and $s_0=6.770732145\ldots$.
This exponent imposes specific denominator divisibility conditions;
**$s_0$ is not an upper bound for the ordinary irrationality measure $\mu$.**
The source proves growth $\log|q_N|/N\to27.86755317\ldots$ and
$\limsup\log|q_N\zeta(s)-p_N^{(s)}|/N=-19.10095491\ldots$ for both readouts.

**Full CMF:** rank-3 ${}_4F_3(1)$, all seven independent parameter axes.
**Start:** $(1,1,1,0;1,1,1)$;
**direction:** $(10,9,8,-5;1,2,3)$.
The walk depth $N$ is the source index, with no offset.
If $(w_0,w_1,w_2)^T$ is the first column of the direct walk, the two readouts are
$$\frac{p_N^{(2)}}{q_N}=\frac{2w_1-w_2}{2w_0},\qquad
\frac{p_N^{(3)}}{q_N}=\frac{w_2}{2w_0}.$$
The common normalization is
$$f_N=\frac{(-1)^N(9N)!(10N)!}{N!(2N)!(3N)!(5N)!(8N)!},
\quad(q_N,p_N^{(2)},p_N^{(3)})=f_N(w_0,w_1-w_2/2,w_2/2).$$
In particular the denominator matches the terminating ${}_4F_3$ in (27).
The paper proves that the two constructions share a third-order recurrence.

`local_checks/verify_dauguet_zudilin2014.py` independently expands the defining rational
functions (5) and (15), specialized by (25) and (28), into partial fractions.
It checks both normalized numerator sequences, the shared denominator and the
finite sum (27) at indices 0–10, all axis pairs at two off-trajectory points,
and the depth-1000 errors at two Arb precisions. The source's prime-divisibility
argument is referenced, not re-proved by this computation.

**Source discrepancy:** the displayed initial-value line on p. 15 prints
$q_1=12307565655$ and $q_2=5669931265166541788415$.
Both defining constructions and finite sum (27) instead give
$q_1=12307605345$ and $q_2=5669931265185834471825$.
We use these formula-derived values; the printed numerator values agree.
The overall sign is normalized to (27), which cancels in the rational readouts.
For $\zeta(3)$, the verified finite-depth exponent is $\delta_{1000}=0.039068352920$.


In [ ]:
cmf = pFq(4, 3, 1)
axes = sp.symbols("x0:4") + sp.symbols("y0:3")
start = dict(zip(axes, [1, 1, 1, 0, 1, 1, 1]))
trajectory = dict(zip(axes, [10, 9, 8, -5, 1, 2, 3]))
depths = list(range(10, 1001, 10))
walks = cmf.walk(trajectory, depths, start)
approximants = [W[2,0]/(2*W[0,0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Dauguet–Zudilin (2014)"] = (depths, deltas)
print(f"delta at source index 1000: {deltas[-1]:.12f}")


delta at source index 1000: 0.039068352920


## 4. Dvornicich–Viola (1987; proceedings 1990)

R. Dvornicich and C. Viola, *Some remarks on Beukers' integrals*,
Colloq. Math. Soc. János Bolyai **51**, 637–657 · $\mu(\zeta(3))\le12.7436$.
The construction on p. 650 of [the original paper](https://mek.oszk.hu/24100/24107/pdf/24107_2.pdf)
inserts $R(f)=f^{2k}(1-34f)^{2n}(1-35f)^{2m}$ into Beukers' triple integral, with
$$f(x,y,z)=\frac{x(1-x)y(1-y)z(1-z)}{1-(1-xy)z},$$
$k=\lfloor\lambda m\rfloor$, $n=\lfloor\nu m\rfloor$ and the printed rounded slopes
$(\lambda,\nu)=(56.41,1.342)$.

**CMF:** $D_3(e_0,e_1,e_2)$, the moment CMF of
$S(e_0,e_1,e_2)=\iiint f^{e_0}(1-34f)^{e_1}(1-35f)^{e_2}\,dx\,dy\,dz/(1-(1-xy)z)$
(`dvornicich_viola_zeta3`, rank 8) · **Start:** $(0,0,0)$ ·
**Path:** the staircase $(2\lfloor56.41m\rfloor,2\lfloor1.342m\rfloor,2m)$, $m=10,20,\ldots,300$ ·
$S=C+Z\zeta(3)$, $p_m/q_m=-C_0/Z_0$.

The moments $J_\ell=S(\ell,0,0)$ are Beukers' integrals, $J_\ell=2(a_\ell\zeta(3)-b_\ell)$ with
Apéry's numbers; $J_0=2\zeta(3)$, $J_1=10\zeta(3)-12$, and
$$(\ell+1)^3J_{\ell+1}-(34\ell^3+51\ell^2+27\ell+5)J_\ell+\ell^3J_{\ell-1}=0\qquad(\ell\ge1).$$
This gives one recurrence in $e_0$ at every $(e_1,e_2)$; $M_{e_0}$ is its companion matrix,
$M_{e_1}=I-34M_{e_0}$ and $M_{e_2}=I-35M_{e_0}$. The state is
$(S(e_0,e_1,e_2),\ldots,S(e_0+7,e_1,e_2))$. As for $\zeta(2)$, the walk steps one axis at a time
through the staircase points and multiplies the steps between two samples as a balanced tree.
The finite-depth exponent below is a numerical quantity, not the published bound.

In [4]:
from flint import fmpq, fmpq_mat

from cmfs import axis_steps, dvornicich_viola_zeta3

e0, e1, e2 = sp.symbols("e0 e1 e2")
M = axis_steps(dvornicich_viola_zeta3())  # M(x, point): the axis matrix M_x at an integer point
# Apery's numbers for zeta(3): J_l = 2 (a_l zeta(3) - b_l).
a, b = [1, 5], [fmpq(0), fmpq(6)]
for l in range(1, 1200):  # enough terms for the readout check below
    a.append(((34 * l**3 + 51 * l**2 + 27 * l + 5) * a[l] - l**3 * a[l - 1]) // (l + 1) ** 3)
    b.append(((34 * l**3 + 51 * l**2 + 27 * l + 5) * b[l] - l**3 * b[l - 1]) / (l + 1) ** 3)
# The rows C and Z of the start state (J_0, ..., J_7), with J_l = C_l + Z_l zeta(3).
dv_start = fmpq_mat([[-2 * b[l] for l in range(8)], [2 * a[l] for l in range(8)]])


def staircase(m):
    """The point (2 floor(56.41 m), 2 floor(1.342 m), 2m) of the Dvornicich--Viola path."""
    return {e0: 2 * (5641 * m // 100), e1: 2 * (1342 * m // 1000), e2: 2 * m}

def product(matrices):
    """The ordered product of `matrices`, multiplied pairwise as a balanced tree."""
    while len(matrices) > 1:
        matrices = [matrices[i] * matrices[i + 1] if i + 1 < len(matrices) else matrices[i]
                    for i in range(0, len(matrices), 2)]
    return matrices[0]

dv_depths = list(range(10, 301, 10))
state, point, previous = dv_start, staircase(0), 0
approximants = []
for m in dv_depths:
    steps = []
    for j in range(previous + 1, m + 1):
        target = staircase(j)
        for x in (e0, e1, e2):
            while point[x] < target[x]:
                steps.append(M(x, point))
                point[x] += 1
    state, previous = state * product(steps), m
    approximants.append(-state[0, 0] / state[1, 0])
deltas = [delta(r) for r in approximants]
results["Dvornicich–Viola (1990)"] = (dv_depths, deltas)
print(f"delta at m = 300: {deltas[-1]:.9f}")

delta at m = 300: 0.085208154


**Readout check.** If $(1-34t)^{e_1}(1-35t)^{e_2}=\sum_jr_jt^j$, then
$S(e_0,e_1,e_2)=\sum_jr_jJ_{e_0+j}$. The next cell computes these sums directly from Apéry's numbers
and checks that they equal the first column of the walked state, exactly, at the first 10 staircase
points, independently of the CMF matrices.

In [5]:
t = sp.Symbol("t")
state, point = dv_start, staircase(0)
for m in range(1, 11):
    target = staircase(m)
    for x in (e0, e1, e2):
        while point[x] < target[x]:
            state = state * M(x, point)
            point[x] += 1
    r = sp.Poly((1 - 34 * t) ** target[e1] * (1 - 35 * t) ** target[e2], t).all_coeffs()[::-1]
    l = target[e0]
    C = sum(-2 * int(r_j) * b[l + j] for j, r_j in enumerate(r))
    Z = sum(fmpq(2 * int(r_j) * a[l + j]) for j, r_j in enumerate(r))
    assert state[0, 0] == C and state[1, 0] == Z, m
print("walk = direct moment sums at m = 1, ..., 10")

walk = direct moment sums at m = 1, ..., 10


## 5. Hata (2000)

M. Hata, *A new irrationality measure for $\zeta(3)$*, Acta Arith.
**92** (2000), 47–57. The paper proves $\mu(\zeta(3))<7.377956$.

**CMF:** Meijer $G$ · **Start:** $(1,1,2,2;1,1,1,1)$ ·
**Direction:** $(0,1,15,16;8,7,8,9)$. We use the direct walk and the
extraction $p_N/q_N=s_1/(4s_0+2s_1)$.

In [6]:
cmf = MeijerG(4, 2, 4, 4, 1)
start = {a0: 1, a1: 1, a2: 2, a3: 2, b0: 1, b1: 1, b2: 1, b3: 1}
trajectory = {a0: 0, a1: 1, a2: 15, a3: 16, b0: 8, b1: 7, b2: 8, b3: 9}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [W[1, 0] / (4 * W[0, 0] + 2 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Hata (2000)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.156407426


## 6. Rhin–Viola (2001)

G. Rhin and C. Viola, *The group structure for $\zeta(3)$*,
Acta Arith. **97** (2001), 269–293.
[Paper](https://www.impan.pl/shop/en/publication/transaction/download/product/82430).
The paper proves $\mu(\zeta(3))<5.513891$.

**CMF:** Meijer $G$ · **Start:** $(1,1,2,2;1,1,1,1)$ ·
**Direction:** $(0,7,31,32;18,17,16,19)$. We use the direct walk and the
extraction $p_N/q_N=s_1/(4s_0+2s_1)$, as for Hata.

In [7]:
cmf = MeijerG(4, 2, 4, 4, 1)
start = {a0: 1, a1: 1, a2: 2, a3: 2, b0: 1, b1: 1, b2: 1, b3: 1}
trajectory = {a0: 0, a1: 7, a2: 31, a3: 32, b0: 18, b1: 17, b2: 16, b3: 19}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [W[1, 0] / (4 * W[0, 0] + 2 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Rhin–Viola (2001)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.221833369


## 7. Giraudin (2026 preprint)

D. Niedbala Giraudin, [*A new upper bound for the irrationality exponent of $\zeta(3)$*](https://arxiv.org/abs/2609.26980) ·
$\mu(\zeta(3))<5.5138800$. The bound applies Rhin–Viola's Theorem 5.1 to the point
$$(h,j,k,l,m,q,r,s)=(832002,883999,987993,780005,624014,572017,468023,676011)$$
of the Rhin–Viola integral
$$I(h,j,k,l,m,q,r,s)=\int_{[0,1]^3}\frac{x^h(1-x)^ly^k(1-y)^sz^j(1-z)^q}
{(1-(1-xy)z)^{q+h-r+1}}\,dx\,dy\,dz,$$
which is $52000\cdot(16,17,19,15,12,11,9,13)$ shifted by $(2,-1,-7,5,14,17,23,11)$.

**Coordinate conversion.** Writing $1-(1-xy)z=(1-z)+xyz$ and expanding its power as a
Mellin–Barnes integral, the three beta integrals give
$$I=\frac{l!\,s!}{(q+h-r)!\,(j+r-h)!}\,
G^{4,2}_{4,4}\!\left(1\,\middle|\,\begin{matrix}1,\ 1+h-r,\ h+l+2,\ k+s+2\\
q+h-r+1,\ j+1,\ h+1,\ k+1\end{matrix}\right).$$
So the point $n\cdot(h,\ldots,s)$ is the Meijer $G$ point $x_0+n\,v$ with
$x_0=(1,1,2,2;1,1,1,1)$ and $v=(0,h-r,h+l,k+s;q+h-r,j,h,k)$. For Rhin–Viola's
$(16,17,19,15,12,11,9,13)$ this is the direction $(0,7,31,32;18,17,16,19)$ of
Section 3, and for Giraudin's point it is
$$v=(0,363979,1612007,1664004;935996,883999,832002,987993)
=52000\cdot(0,7,31,32;18,17,16,19)+(0,-21,7,4;-4,-1,2,-7).$$

**CMF:** Meijer $G$ · **Start:** $(1,1,2,2;1,1,1,1)$ · **Direction:** $v$ above, depth $N=1$ ·
direct walk, extraction $p_N/q_N=s_1/(4s_0+2s_1)$, as for Rhin–Viola.
One step of $v$ is about $7\cdot10^6$ axis steps, so the walk takes $52000$ steps of the
Rhin–Viola trajectory, multiplied as a balanced tree, and then the remaining $46$ axis steps.
Every axis matrix vanishes on the hyperplane $\sum a-\sum b=3$; the Rhin–Viola line has
$\sum a-\sum b=2$, and the remaining steps alternate so that they never reach it.
At $N=1$, $q_N$ has about $1.8$ million digits. The finite-depth exponent is a numerical
quantity, not the published bound, which also uses Rhin–Viola's arithmetic Theorem 5.1.

In [ ]:
# Giraudin's point in the Rhin--Viola integral and its Meijer G direction.
h, j, k, l, m, q, r, s = 832002, 883999, 987993, 780005, 624014, 572017, 468023, 676011
assert h + m == k + r and j + q == l + s
axes = (a0, a1, a2, a3, b0, b1, b2, b3)
direction = (0, h - r, h + l, k + s, q + h - r, j, h, k)
rhin_viola = (0, 7, 31, 32, 18, 17, 16, 19)
shift = dict(zip(axes, (v - 52000 * w for v, w in zip(direction, rhin_viola))))
print("direction:", direction, " = 52000 * Rhin-Viola +", tuple(shift.values()))

cmf = MeijerG(4, 2, 4, 4, 1)
start = dict(zip(axes, (1, 1, 2, 2, 1, 1, 1, 1)))
T = cmf.trajectory_matrix(dict(zip(axes, rhin_viola)), start)  # one Rhin--Viola step, in n
M = axis_steps(cmf)  # M(x, point): the axis matrix M_x at an integer point


def compiled(matrix, x):
    """`matrix` as a function of the integer x, with exact python-flint entries."""
    entries = []
    for row in matrix.tolist():
        compiled_row = []
        for entry in row:
            p, q = sp.fraction(sp.cancel(sp.together(entry)))
            cp, p = sp.Poly(p, x).clear_denoms()
            cq, q = sp.Poly(q, x).clear_denoms()
            scale = sp.Rational(cq) / sp.Rational(cp)
            compiled_row.append((sp.lambdify(x, p.as_expr(), modules=[{}]),
                                 sp.lambdify(x, q.as_expr(), modules=[{}]),
                                 fmpq(int(scale.p), int(scale.q))))
        entries.append(compiled_row)
    return lambda value: fmpq_mat([[c * fmpq(int(p(value)), int(q(value))) for p, q, c in row] for row in entries])


T = compiled(T, n)
# 52000 Rhin--Viola steps, multiplied as a balanced tree (a binary counter of partial products).
stack = []
for i in range(52000):
    product, size = T(i), 1
    while stack and stack[-1][0] == size:
        product, size = stack.pop()[1] * product, 2 * size
    stack.append((size, product))
W = stack[0][1]
for _, product in stack[1:]:
    W = W * product

# The remaining axis steps, alternating between steps that lower and raise
# sum(a) - sum(b) - 3, which is -1 on the Rhin--Viola line; every axis matrix is singular where it is 0.
point = {x: start[x] + 52000 * w for x, w in zip(axes, rhin_viola)}
sigma = lambda p: sum(p[x] for x in axes[:4]) - sum(p[x] for x in axes[4:]) - 3
lower = [x for x in axes for _ in range(abs(shift[x])) if (shift[x] < 0) == (x in axes[:4])]
higher = [x for x in axes for _ in range(abs(shift[x])) if (shift[x] < 0) != (x in axes[:4])]
for x in (y for pair in zip(lower, higher) for y in pair):
    if shift[x] > 0:
        assert sigma(point) != 0
        W = W * M(x, point)
        point[x] += 1
    else:
        point[x] -= 1
        assert sigma(point) != 0
        W = W * M(x, point).inv()
assert all(point[x] == start[x] + v for x, v in zip(axes, direction))

approximant = W[1, 0] / (4 * W[0, 0] + 2 * W[1, 0])
p_N, q_N = int(approximant.p), int(approximant.q)
with ctx.workprec(2 * q_N.bit_length() + 256):  # beyond the precision of ZETA3 above
    error = abs(arb(3).zeta() - arb(p_N) / q_N)
    giraudin_delta = float((-1 - error.log() / arb(q_N).log()).mid())
print(f"q_N has {len(str(q_N))} digits")
print(f"delta at depth 1: {giraudin_delta:.9f}")

direction: (0, 363979, 1612007, 1664004, 935996, 883999, 832002, 987993)  = 52000 * Rhin-Viola + (0, -21, 7, 4, -4, -1, 2, -7)


q_N has 1767961 digits
delta at depth 1: 0.221488328


## 8. New Meijer $G$ trajectory (this work)

The $\zeta(3)$ readout of the simultaneous $\zeta(3),\zeta(4)$ construction of
the paper (Section 6). The paper proves $\delta>0.22521567$ and
$\mu(\zeta(3))<5.44019$ for it.

**CMF:** Meijer $G$, $G^{5,2}_{5,5}(1)$ · **Start:** $(1,1,0,0,2;0,0,0,0,0)$ ·
**Direction:** $(11,12,-16,-17,10;0,1,2,-1,-2)$. We use the inverse-transpose
walk $(s_0,s_1,s_2,\ldots)$ and the extraction
$p_N/q_N=(-3s_1+3s_2)/(6s_2)$, the first and third rows of
$$\mathcal R_N=\begin{pmatrix}0&-3&3\\2&-2&0\\0&0&6\end{pmatrix}
\begin{pmatrix}s_0\\s_1\\s_2\end{pmatrix}.$$

In [8]:
cmf = MeijerG(5, 2, 5, 5, 1)
a = sp.symbols("a0:5")
b = sp.symbols("b0:5")
start = dict(zip(a + b, [1, 1, 0, 0, 2, 0, 0, 0, 0, 0]))
trajectory = dict(zip(a + b, [11, 12, -16, -17, 10, 0, 1, 2, -1, -2]))
# The dual walk is the inverse transpose of the walk, inverted numerically (FLINT).
matrix = cmf.trajectory_matrix(trajectory, start)
walks = NumericMatrix.walk(matrix, Position({n: 1}), DEPTHS, Position({n: 0}))
duals = [W.inv().transpose() for W in walks]
rows = [(-3 * D[1, 0] + 3 * D[2, 0], 2 * D[0, 0] - 2 * D[1, 0], 6 * D[2, 0]) for D in duals]
approximants = [p3 / q for p3, p4, q in rows]
deltas = [delta(r) for r in approximants]
results["New Meijer $G$ trajectory"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.226506885


## 9. Simultaneous approximations to $\zeta(3)$ and $\zeta(4)$

The same walk approximates $\zeta(4)$ through the second row of $\mathcal R_N$,
$p^{(4)}_N/q_N=(2s_0-2s_1)/(6s_2)$, with the same denominator. For a fixed
rational pair $(A,B)\ne(0,0)$ the readout of $A\zeta(3)+B\zeta(4)$ is
$(Ap^{(3)}_N+Bp^{(4)}_N)/q_N$. The paper proves $\delta>0.015629$ and
$\mu(A\zeta(3)+B\zeta(4))<64.983$ for every such pair, so $1,\zeta(3),\zeta(4)$
are linearly independent over $\mathbb Q$. Below we take $(A,B)=(15,1)$, the
combination ranked in the local neighbourhood search. Here
$\delta_N$ is measured against the target of each readout.

In [ ]:
joint_results = {
    "ζ(3)": (DEPTHS, deltas),
    "ζ(4)": (DEPTHS, [delta(p4 / q, ZETA4) for p3, p4, q in rows]),
    "15ζ(3) + ζ(4)": (DEPTHS, [delta((15 * p3 + p4) / q, JOINT) for p3, p4, q in rows]),
}
for name, (depths, values) in joint_results.items():
    print(f"{name:>13}  delta at depth 3000: {values[-1]:.9f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in joint_results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Hata (1990): integral identified, CMF still unresolved

Theorem 5.3 of [Hata's paper](https://doi.org/10.1515/crll.1990.407.99)
uses two Legendre-type polynomials and a rational correction to its triple
integral. On the subsequence $n=9N$, integration by parts gives the
following small linear form, up to the explicit factorial/divisor
normalization in that proof:
$$\int_{[0,1]^3}
\frac{x^{8N}(1-x)^{10N}y^{7N}(1-y)^{9N}v^{8N}(1-v)^{9N}}
{[1-x(1-v)]^{9N+1}(1-yv)^{8N+1}}\,dx\,dy\,dv.$$
Integrating $x,y$ expresses it as the beta-normalized convolution
$$B(8N+1,10N+1)B(7N+1,9N+1)
\int_0^1v^{8N}(1-v)^{9N}
{}_2F_1(9N+1,8N+1;18N+2;1-v)
{}_2F_1(8N+1,7N+1;16N+2;v)\,dv.$$
This supplies a concrete starting kernel for a larger field. A commuting
set of rational shift matrices and the corrected rational extraction have
not yet been constructed, so the historical row remains unverified.


## All constructions

$\delta_N$ against the walk depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published upper bound | Notebook coverage |
|:--|--:|:--|
| Apéry (1978/1979) | $13.41782$ | Full CMF; source $N=1000$, $\delta=0.085098239651$ |
| Zudilin (2002), simultaneous | Convergence; no irrationality bound | Full CMF; source $N=1000$, $\delta=-0.038940836181$ |
| Dauguet–Zudilin (2014), simultaneous | $s_{\tau_0}\le6.770732145\ldots$ (not $\mu$) | Full CMF; $\delta_{1000}=0.039068352920$ |
| Dvornicich–Viola (1987) | $12.7436$ | CMF above ($m=300$) |
| Hata (1990) | $8.83028$ | Integral identified above; CMF unresolved |
| Hata (2000) | $7.377956$ | CMF above |
| Rhin–Viola (2001) | $5.513891$ | CMF above |
| Giraudin (2026) | $5.5138800$ | CMF above ($N=1$) |
| This work: $\zeta(3)$ | $5.44019$ | CMF above |
| This work: $A\zeta(3)+B\zeta(4)$ | $64.983$ | Same CMF above |

The earlier bounds are summarized in the introduction to
[Rhin–Viola (2001)](https://www.impan.pl/shop/en/publication/transaction/download/product/82430).
Rows marked “reference only” have no verified CMF encoding in the supplied material.
